In [1]:
import sys
import os
from pathlib import Path

# Walk upward until we find the folder that contains "src"
p = Path.cwd().resolve()
while p != p.parent and not (p / "src").exists():
    p = p.parent

if not (p / "src").exists():
    raise RuntimeError("Could not find repo root (folder containing 'src').")

if str(p) not in sys.path:
    sys.path.insert(0, str(p))

print("Repo root set to:", p)


Repo root set to: /home/iauger/projects/dsci632-project


In [2]:
import os, subprocess
import sys, platform

print("python:", sys.executable)
print("platform:", platform.platform())
print("cwd:", os.getcwd())
print("home:", str(Path.home()))

assert "/home/" in str(Path.home()), "Not running in WSL home"
assert "mnt/c" not in os.getcwd().lower(), "CWD is on Windows mount; use ~/projects/... instead"
print("JAVA_HOME:", os.environ.get("JAVA_HOME"))
print("which java:", subprocess.check_output(["which","java"]).decode().strip())

python: /home/iauger/projects/dsci632-project/.venv/bin/python
platform: Linux-5.15.167.4-microsoft-standard-WSL2-x86_64-with-glibc2.39
cwd: /home/iauger/projects/dsci632-project/notebooks
home: /home/iauger
JAVA_HOME: /usr/lib/jvm/java-17-openjdk-amd64
which java: /usr/lib/jvm/java-17-openjdk-amd64/bin/java


In [3]:
from src.config import load_settings 

s = load_settings()
print("ENV:", s.env)
print("RAW recipes:", s.raw_recipes_path)
print("RAW interactions:", s.raw_interactions_path)
print("PROCESSED:", s.processed_dir)
print("BRONZE:", s.bronze_dir)
print("SILVER:", s.silver_dir)
print("GOLD:", s.gold_dir)
print("RUN_ID:", s.features_run_id)
print("FEATURES DATASET",s.features_dataset_path)

ENV: local
RAW recipes: /home/iauger/projects/dsci632-project/data/raw/RAW_recipes.csv
RAW interactions: /home/iauger/projects/dsci632-project/data/raw/RAW_interactions.csv
PROCESSED: /home/iauger/projects/dsci632-project/data/processed
BRONZE: /home/iauger/projects/dsci632-project/data/processed/bronze
SILVER: /home/iauger/projects/dsci632-project/data/processed/silver
GOLD: /home/iauger/projects/dsci632-project/data/processed/gold
RUN_ID: 20260227_152815
FEATURES DATASET /home/iauger/projects/dsci632-project/data/processed/features/runs/20260227_152815/dataset.parquet


In [4]:
from src.spark.session import get_spark
    
spark = get_spark(app_name="testing-spark-session", debug=True, )

print("spark.version:", spark.version)
print("spark.driver.host:", spark.conf.get("spark.driver.host", "NOT SET"))
print("spark.local.ip:", spark.conf.get("spark.local.ip", "NOT SET"))

your 131072x1 screen size is bogus. expect trouble
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/28 08:33:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/02/28 08:33:32 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


spark.version: 3.5.1
spark.driver.host: 127.0.0.1
spark.local.ip: NOT SET


In [5]:
# Block 1: Marathon Build - Semi-Supervised Learning (1M+ Reviews)
import time
import psutil
from src.spark.features.build_features import build_features_with_full_corpus_w2v
from src.config import load_settings
from src.spark.labeling.taxonomy import get_tag_ids

# 1. Initialize Settings
s = load_settings()
print(f"Starting Full Corpus Build: {s.features_run_id}")
print(f"CPU/Memory: {psutil.cpu_count()} cores | {psutil.virtual_memory().total / (1024**3):.1f}GB RAM")

# 2. Execute the Build
t0 = time.time()
try:
    # This fits Word2Vec on the full 1.4M corpus and calibrates thresholds on silver labels
    build_features_with_full_corpus_w2v(spark, labels=get_tag_ids())
    
    elapsed = time.time() - t0
    print(f"\nBuild Successful!")
    print(f"Total Time: {elapsed / 60:.2f} minutes")
    print(f"Dataset saved to: {s.features_dataset_path}")

except Exception as e:
    print(f"\nBuild Failed!")
    print(f"Error: {str(e)}")

# 3. Quick Threshold Preview (Algorithm 5)
import json
from pathlib import Path

metrics_path = Path(s.features_metrics_path)
if metrics_path.exists():
    with open(metrics_path, "r") as f:
        metrics = json.load(f)
    
    print("\n--- Calibrated Thresholds (Algorithm 5 Diagonal) ---")
    thresholds = metrics.get("tag_thresholds", {})
    # Sort by threshold value to see semantic density
    for tag, val in sorted(thresholds.items(), key=lambda x: x[1], reverse=True):
        print(f"{tag:.<35} {val:.4f}")

Starting Full Corpus Build: 20260227_131050
CPU/Memory: 6 cores | 11.7GB RAM


26/02/27 15:29:30 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS
26/02/27 17:59:07 WARN TaskSetManager: Stage 51 contains a task of very large size (33477 KiB). The maximum recommended task size is 1000 KiB.
26/02/27 17:59:16 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.



Build Successful!
Total Time: 151.09 minutes
Dataset saved to: /home/iauger/projects/dsci632-project/data/processed/features/runs/20260227_131050/dataset.parquet

--- Calibrated Thresholds (Algorithm 5 Diagonal) ---
too_acidic......................... 0.8221
mushy_soggy........................ 0.7701
too_salty.......................... 0.7384
dry................................ 0.7370
too_sweet.......................... 0.7273
time_consuming_complex............. 0.7140
too_spicy.......................... 0.7000
would_not_make_again............... 0.6830
bland_lacks_flavor................. 0.6807
ingredient_issue................... 0.6576
moist_tender....................... 0.6568
crispy_crunchy..................... 0.6480
substitution_modification.......... 0.6469
easy_quick......................... 0.6206
would_make_again................... 0.6070
delicious_tasty.................... 0.6069
family_hit......................... 0.6042


In [11]:
from pyspark.ml.feature import Word2VecModel

# Explicitly point to the successful run shown in your VS Code explorer
marathon_model_path = "/home/iauger/projects/dsci632-project/data/processed/features/runs/20260227_152815/pipeline_models/w2v_model"
model = Word2VecModel.load(marathon_model_path)

print("🔍 Verifying synonyms from the 1.4M review corpus:")
model.findSynonyms("salty", 5).show()
model.findSynonyms("confusing", 5).show()

🔍 Verifying synonyms from the 1.4M review corpus:
+-------+------------------+
|   word|        similarity|
+-------+------------------+
| liking|0.7422865629196167|
|  bland| 0.715444803237915|
|saltier|0.7141106724739075|
|   oily| 0.702180802822113|
|    tad|0.7019623517990112|
+-------+------------------+

+---------+------------------+
|     word|        similarity|
+---------+------------------+
| confused|0.8987341523170471|
|    vague|0.8926295042037964|
|  unclear|0.8469954133033752|
| specific|0.8031492233276367|
|confusion| 0.778710663318634|
+---------+------------------+



In [6]:
# Updated Block 2: Verify Semi-Supervised Context
import json
from pathlib import Path

# 1. Verify the metrics show the 1M+ corpus was used
metrics_path = Path(s.features_metrics_path)
with open(metrics_path, "r") as f:
    metrics = json.load(f)

print(f"📊 Word2Vec Corpus Size: {metrics.get('w2v_corpus_size', 0):,}")
print(f"🏷️ Labeled Calibration Set: {metrics.get('labeled_set_size', 0):,}")

# 2. Test improved synonyms (the "Salty" test)
from pyspark.ml.feature import Word2VecModel
model = Word2VecModel.load(f"{s.features_pipeline_model_dir}/w2v_model")

for word in ["salty", "confusing", "delicious"]:
    print(f"\nSynonyms for '{word}':")
    model.findSynonyms(word, 5).show()

📊 Word2Vec Corpus Size: 0
🏷️ Labeled Calibration Set: 0

Synonyms for 'salty':
+---------+-------------------+
|     word|         similarity|
+---------+-------------------+
|sensitive| 0.5456942319869995|
|   liking| 0.5378662943840027|
|     oily| 0.5083402395248413|
|    claim|0.48956331610679626|
|      tad| 0.4856690764427185|
+---------+-------------------+


Synonyms for 'confusing':
+---------+------------------+
|     word|        similarity|
+---------+------------------+
|measuring|0.6002045273780823|
|  readers| 0.586935818195343|
|   ensure|0.5797135233879089|
|decreased|0.5394229292869568|
|  guessed|0.5372918248176575|
+---------+------------------+


Synonyms for 'delicious':
+---------+-------------------+
|     word|         similarity|
+---------+-------------------+
|   divine|0.44674623012542725|
|  freakin|0.44554582238197327|
|flavorful|0.43301036953926086|
|brilliant|0.42726001143455505|
|wonderful|0.42369508743286133|
+---------+-------------------+



In [12]:
# Updated Block 3: Algorithm 5 Verification
print("--- Data-Driven Thresholds (Normalized) ---")
thresholds = metrics.get("tag_thresholds", {})
for tag, val in sorted(thresholds.items(), key=lambda x: x[1], reverse=True):
    print(f"{tag:.<35} {val:.4f}")

# Verify the "Inverse Threshold" fix
# High-frequency positive tags should no longer have lower thresholds than rare negatives
pos_t = thresholds.get("delicious_tasty", 0)
neg_t = thresholds.get("too_acidic", 0)
print(f"\nDelicious Threshold: {pos_t:.4f}")
print(f"Too Acidic Threshold: {neg_t:.4f}")

--- Data-Driven Thresholds (Normalized) ---
too_acidic......................... 0.8221
mushy_soggy........................ 0.7701
too_salty.......................... 0.7384
dry................................ 0.7370
too_sweet.......................... 0.7273
time_consuming_complex............. 0.7140
too_spicy.......................... 0.7000
would_not_make_again............... 0.6830
bland_lacks_flavor................. 0.6807
ingredient_issue................... 0.6576
moist_tender....................... 0.6568
crispy_crunchy..................... 0.6480
substitution_modification.......... 0.6469
easy_quick......................... 0.6206
would_make_again................... 0.6070
delicious_tasty.................... 0.6069
family_hit......................... 0.6042

Delicious Threshold: 0.6069
Too Acidic Threshold: 0.8221


In [7]:
from pyspark.ml.pipeline import PipelineModel
from pyspark.ml.feature import Word2VecModel
from pyspark.sql import functions as F
from src.spark.features.embeddings import add_word2vec_embeddings, Word2VecSpec
from src.spark.features.io import read_labeled_reviews
from src.spark.features.pipeline import TextFeatureSpec, add_token_union_column, drop_intermediate_columns
from src.spark.labeling.taxonomy import get_tag_ids
from src.spark.features.splits import assign_recipe_splits
from src.spark.features.labels import add_binary_label_cols




# 1. Setup paths from the successful run
s = load_settings(prefer_latest_run=True)
labels = get_tag_ids()

# 2. Load the exact models from the marathon run
prep_model = PipelineModel.load(f"{s.features_pipeline_model_dir}/prep_model")
w2v_model = Word2VecModel.load(f"{s.features_pipeline_model_dir}/w2v_model")

# 3. Reload and Prepare the Labeled Reviews
new_labeled_path = f"{s.processed_dir}/labeling/zero_shot/labeled_gold_reviews_v4.parquet"
labeled_df = spark.read.parquet(new_labeled_path)

# Apply splits and labels to the fresh 17k rows
labeled_df = assign_recipe_splits(labeled_df, recipe_id_col="recipe_id")
labeled_df = add_binary_label_cols(labeled_df, labels)

# 4. Reconstruct df_embed (This is fast, ~1-2 mins)
spec = TextFeatureSpec(text_col="review_clean", output_col="features", token_union_col="tokens_all")
w2v_spec = Word2VecSpec(input_col="tokens_all", output_col="review_embeddings")

df_prep = add_token_union_column(prep_model.transform(labeled_df), spec)
df_embed = add_word2vec_embeddings(df_prep, model=w2v_model, spec=w2v_spec)
# Align column name to "features" as expected by PrototypeSpec
df_embed = df_embed.withColumn("features", F.col("review_embeddings"))

print(f"✅ df_embed reconstructed with {df_embed.count()} labeled rows.")

✅ df_embed reconstructed with 27000 labeled rows.


In [8]:
from src.spark.features.prototypes import PrototypeSpec, build_tag_centroids

p_spec = PrototypeSpec(features_col="features", label_prefix="y_")
centroids_df = build_tag_centroids(df_embed, spec=p_spec, labels=get_tag_ids())

# 3. Save them now so you have them for the future!
centroids_df.write.mode("overwrite").parquet(s.features_tag_centroids_path)

26/02/28 08:42:26 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS
26/02/28 08:42:27 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [9]:
import json

# 1. Load the metrics from the 152815 Marathon Run
with open(s.features_metrics_path, "r") as f:
    metrics = json.load(f)

# 2. Extract the thresholds map and the list of tags
thresholds = metrics["tag_thresholds"]
tags = list(thresholds.keys())

print(f"✅ Loaded {len(tags)} tags from marathon metrics.")
print(f"Tags to process: {tags}")

✅ Loaded 17 tags from marathon metrics.
Tags to process: ['bland_lacks_flavor', 'crispy_crunchy', 'delicious_tasty', 'dry', 'easy_quick', 'family_hit', 'ingredient_issue', 'moist_tender', 'mushy_soggy', 'substitution_modification', 'time_consuming_complex', 'too_acidic', 'too_salty', 'too_spicy', 'too_sweet', 'would_make_again', 'would_not_make_again']


In [10]:
# Block 3: Full Corpus Embedding + Labeling (Algorithm 6)
from pyspark.ml.functions import vector_to_array
from pyspark.sql import functions as F
from src.spark.features.pipeline import add_token_union_column
from src.spark.features.embeddings import add_word2vec_embeddings
from src.spark.modeling.inference import _native_cosine_sim

# 1. Load the 1.4M Clean Reviews (Text only)
raw_corpus = spark.read.parquet(s.silver_interactions_path)

# 2. Transform into Vectors using Marathon Models
# This step turns text into the 'features' vectors used for cosine similarity
df_prep = add_token_union_column(prep_model.transform(raw_corpus), spec)
full_featured_df = add_word2vec_embeddings(df_prep, model=w2v_model, spec=w2v_spec)
full_featured_df = full_featured_df.withColumn("features", F.col("review_embeddings"))

# 3. Labeling Loop (Algorithm 6)
gold_df = full_featured_df
for tag in tags:
    row = centroids_df.filter(F.col("tag") == tag).select("centroid").first()
    if not row: continue
    
    c_vec = row["centroid"] 
    t_val = thresholds[tag]
    
    sim_col = f"sim_{tag}"
    pred_col = f"pred_{tag}"
    
    # Calculate similarity using your native math function
    gold_df = gold_df.withColumn(
        sim_col, 
        _native_cosine_sim(vector_to_array(F.col("features")), F.lit(c_vec))
    )
    gold_df = gold_df.withColumn(pred_col, (F.col(sim_col) >= F.lit(t_val)).cast("int"))

# 4. Clean up heavy columns (tokens/sim scores) to save disk space
final_cols = [c for c in gold_df.columns if not c.startswith("sim_") 
              and c not in ["tokens", "tokens_all", "review_embeddings"]]
gold_df = gold_df.select(*final_cols)

# 5. Save the 1.4M Gold Dataset for the MLP
final_output_path = f"{s.processed_dir}/gold/gold_features_1.4M.parquet"
gold_df.write.mode("overwrite").parquet(final_output_path)

print(f"🎉 SUCCESS! Labeled 1.4 million reviews.")

🎉 SUCCESS! Labeled 1.4 million reviews.


In [11]:
gold_count = gold_df.count()
print(f"Final Gold Dataset Count: {gold_count:,} reviews")

Final Gold Dataset Count: 1,071,520 reviews


In [12]:
# Check the actual prediction columns generated
pred_cols = [c for c in gold_df.columns if c.startswith("pred_")]
print("Generated Prediction Columns:")
print(pred_cols)

Generated Prediction Columns:
['pred_bland_lacks_flavor', 'pred_crispy_crunchy', 'pred_delicious_tasty', 'pred_dry', 'pred_easy_quick', 'pred_family_hit', 'pred_ingredient_issue', 'pred_moist_tender', 'pred_mushy_soggy', 'pred_substitution_modification', 'pred_time_consuming_complex', 'pred_too_acidic', 'pred_too_salty', 'pred_too_spicy', 'pred_too_sweet', 'pred_would_make_again', 'pred_would_not_make_again']


In [13]:
# Block 2: Quality Assurance & Signal Check (Corrected Names)
print("--- Global Label Prevalence (1.4M Corpus) ---")
total_n = gold_df.count()

# Testing a mix of positive, negative, and complexity tags
test_tags = [
    "delicious_tasty", 
    "too_salty", 
    "time_consuming_complex", 
    "ingredient_issue", 
    "would_make_again"
]

for tag in test_tags:
    pred_col = f"pred_{tag}"
    if pred_col in gold_df.columns:
        hits = gold_df.filter(F.col(pred_col) == 1).count()
        percentage = (hits / total_n) * 100
        print(f"{tag:.<30} {hits:,} hits ({percentage:.2f}%)")

print("\n--- Semantic Spot Check (Complexity/Instruction Quality) ---")
# Verifying that 'time_consuming_complex' captures the instructional nuance
(gold_df.filter(F.col("pred_time_consuming_complex") == 1)
 .select("review_clean")
 .limit(5)
 .show(truncate=False))

--- Global Label Prevalence (1.4M Corpus) ---


delicious_tasty............... 511,481 hits (47.73%)


too_salty..................... 28,269 hits (2.64%)


time_consuming_complex........ 130,898 hits (12.22%)


ingredient_issue.............. 328,349 hits (30.64%)


would_make_again.............. 515,094 hits (48.07%)

--- Semantic Spot Check (Complexity/Instruction Quality) ---
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|review_clean                                                                                                                                                            

In [ ]:
print("\n--- Semantic Spot Check (Complexity/Instruction Quality) ---")
# Verifying that 'too_salty' captures the instructional nuance
(gold_df.filter(F.col("pred_too_salty") == 1)
 .select("review_clean")
 .limit(5)
 .show(truncate=False))


--- Semantic Spot Check (Complexity/Instruction Quality) ---


+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|review_clean 

26/03/01 12:31:03 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 26419856 ms exceeds timeout 120000 ms
26/03/01 12:31:03 WARN SparkContext: Killing executors is not supported by current scheduler.
26/03/01 12:31:04 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint

In [14]:
# 1. Trigger the massive 1.4M write (this takes 20-40 mins)
gold_df.write.mode("overwrite").parquet(final_output_path)

# 2. After it finishes, check the real global scale
final_gold_df = spark.read.parquet(final_output_path)
global_n = final_gold_df.count()
print(f"Total Rows in Final Gold: {global_n:,}")

# Now run the prevalence check on the actual file
hits = final_gold_df.filter(F.col("pred_delicious_tasty") == 1).count()
print(f"Total Delicious Hits: {hits:,} ({(hits/global_n)*100:.2f}%)")

Total Rows in Final Gold: 1,071,520
Total Delicious Hits: 511,481 (47.73%)
